In [1]:
import os
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_KEY')
os.environ["TAVILY_API_KEY"] = os.getenv('TAVILY')

TypeError: str expected, not NoneType

In [2]:
from langchain_openai import ChatOpenAI
model = ChatOpenAI(temperature=0)


## Create tools
Multiply

Search

In [7]:
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults

@tool
def multiply(first_number:int, second_number: int)->int:
    """Multiplies two interger together."""
    return first_number * second_number

@tool
def search(query:   str):
    """perform web search on the user query"""
    tavily = TavilySearchResults(max_results=1)
    result = tavily.invoke(query)
    return result 

In [8]:
# Combine the tools and Bind to the LLM
tools = [search, multiply]
tool_map = {tool.name: tool for tool in tools}
tool_map

{'search': StructuredTool(name='search', description='perform web search on the user query', args_schema=<class 'langchain_core.utils.pydantic.search'>, func=<function search at 0x000001DEE11FEB90>),
 'multiply': StructuredTool(name='multiply', description='Multiplies two interger together.', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x000001DEE11FEE60>)}

In [9]:
model_with_tools = model.bind_tools(tools)


Checking the tool calling with Specific questions

In [15]:
res = model_with_tools.invoke('Search from internet and tell me who won the T20 Cricket world cup in 2024?')
for i in res.additional_kwargs.get("tool_calls", []):
    print(i)


In [14]:
tool_map[i['function']['name']].invoke(i['function']['arguments'])

NameError: name 'i' is not defined

In [16]:
res = model_with_tools.invoke('What is 24*365')
for i in res.additional_kwargs.get("tool_calls", []):
    print(i)


Graph creation

In [20]:
from typing import TypedDict, Annotated, Sequence
import operator, json
from langchain_core.messages import BaseMessage

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

def invoke_model(state):
    messages = state['messages']
    question = messages[-1]
    return {"messaes": [model_with_tools.invoke(question)]}

##### Bring human in the loop
-- Bring human approval for costly tool calls or where we need validation

In [21]:
def invoke_tool(state):
    tool_calls = state['messages'][-1].addtional_kwargs.get("tool_calls", [])

    for tool_call in tool_calls:
        tool_details = tool_call
    
    if tool_details is None:
        raise Exception("No user input found")
    
    print(f'Selected tool: {tool_details.get("function").get("name")}')

    if tool_details.get("function").get("name") == "search":
        response = input(prompt=f"[y/n] continue with expensive web search?")
        if response == "n":
            raise ValueError
        
    response = tool_map[tool_details['function']['name']].invoke(json.loads(tool_details.get("function").get("arguments")))
    return {"messages" : [response]}

In [22]:
from langgraph.graph import StateGraph,END

graph = StateGraph(AgentState) ### StateGraph with AgentState
graph.add_node("agent", invoke_model)
graph.add_node("tool", invoke_tool)

In [27]:
from langgraph.graph import StateGraph,END

graph = StateGraph(AgentState) ### StateGraph with AgentState

graph.add_node("agent", invoke_model)

graph.add_node("tool", invoke_tool)

def router(state):
    tool_calls = state['messages'][-1].additional_kwargs.get("tool_calls", [])
    if len(tool_calls):
        return "tool"
    else:
        return "end"

graph.add_conditional_edges("agent", router, {
    "tool": "tool",
    "end": END,
})

graph.add_edge("tool", END)

graph.set_entry_point("agent")
app = graph.compile()


In [30]:
from IPython.display import Image, display
display(Image(app.get_graph().draw_png()))

ImportError: Install pygraphviz to draw graphs: `pip install pygraphviz`.


Testing with Questions

In [32]:

for s in app.stream({"messages": ["What is 24 * 365?"]}):
    print(list(s.values())[0])
    print("----")

AttributeError: 'str' object has no attribute 'additional_kwargs'

In [33]:

for s in app.stream({"messages": ["Who won the T20 Cricket world cup in 2024?"]}):
    print(list(s.values())[0])
    print("----")

AttributeError: 'str' object has no attribute 'additional_kwargs'

In [34]:


for s in app.stream({"messages": ["Who won the T20 Cricket world cup in 2022?"]}):
    print(list(s.values())[0])
    print("----")
     

AttributeError: 'str' object has no attribute 'additional_kwargs'